# 02 — KAIST Acoustic EDA (Exploratory Data Analysis)

**EDA-only / read-only dataset validation.** No model training, no train/test split, no modification of production DSP/backend ingestion.

Covers: dataset overview, distribution, raw waveforms, amplitude distributions, time-domain statistics (RMS, std, peak-to-peak, kurtosis, skewness, crest factor), FFT, PSD, dominant frequencies, STFT spectrograms, normal vs faulty comparisons, feature distributions, and data-quality observations. Units/frequency axes labeled when verified from file metadata.


In [ ]:
import os, sys, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal

warnings.filterwarnings('ignore', category=RuntimeWarning)
plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['axes.grid'] = True

EDA_DIR = Path.cwd()
if not (EDA_DIR / 'kaist_acoustic_eda_utils.py').exists():
    EDA_DIR = Path.cwd() / 'backend' / 'eda'
sys.path.insert(0, str(EDA_DIR))
from kaist_acoustic_eda_utils import (
    get_acoustic_dir, build_inventory, load_acoustic_values,
    time_domain_stats, fft_spectrum, dominant_frequencies
)

os.environ.setdefault('KAIST_DATA_ROOT', r'G:\My Drive\IndustrialSense\datasets\kaist')
ac_dir = get_acoustic_dir()
out_fig = EDA_DIR / 'outputs' / 'figures'
out_fig.mkdir(parents=True, exist_ok=True)
inv = build_inventory(ac_dir)
files = sorted([p for p in ac_dir.rglob('*.mat') if p.is_file()]) if ac_dir.exists() else []
print('Acoustic dir:', ac_dir)
print('MAT files:', [p.name for p in files])
inv[['filename','fault_category','operating_condition_load','severity_or_variant','inspection_status','sampling_frequency_hz_verified','duration_s_verified','units_verified','warnings']]


## 1. Dataset overview and recording distribution


In [ ]:
dist_fault = inv['fault_category'].value_counts(dropna=False)
dist_cond = inv['operating_condition_load'].value_counts(dropna=False)
dist_pair = inv.groupby(['operating_condition_load','fault_category','severity_or_variant'], dropna=False).size().reset_index(name='count')
print('By fault category:\n', dist_fault)
print('\nBy operating condition/load:\n', dist_cond)
print('\nBy condition/fault/severity:\n', dist_pair)

fig, axes = plt.subplots(1, 2, figsize=(10,4))
dist_fault.plot(kind='bar', ax=axes[0], color=['#4c78a8','#f58518','#54a24b'])
axes[0].set_title('Recording distribution by fault category')
axes[0].set_xlabel('Fault category (filename-derived)')
axes[0].set_ylabel('Count')
dist_cond.plot(kind='bar', ax=axes[1], color='#72b7b2')
axes[1].set_title('Recording distribution by operating condition/load')
axes[1].set_xlabel('Operating condition/load (filename-derived)')
axes[1].set_ylabel('Count')
plt.tight_layout(); plt.savefig(out_fig/'distribution.png', dpi=150); plt.show()
dist_pair


## 2. Per-file analysis (processed sequentially; full signals not retained simultaneously)

Signals: `Signal.y_values.values` (verified). Time axis: `Signal.x_values` (`s`, fs = 1/increment when verified). Amplitude unit: verified from `y_values.quantity.label` (e.g. `Pa`).


In [ ]:
records = []  # per-file stats/features
segments = {}  # short segments for waveform/PSD/spectrogram plots
MAX_PLOT_SEC = 0.2  # short segment for visibility
PSD_NPERSEG = 4096

for p in files:
    y, meta = load_acoustic_values(p)
    fs = meta['fs_hz']
    unit = meta['y_unit']
    # time-domain stats
    st = time_domain_stats(y)
    st.update({'file': p.name, 'fs_hz': fs, 'y_unit': unit, 'n_samples': len(y), 'duration_s': meta['duration_s']})
    # dominant freqs (full signal FFT can be heavy; use segment for responsiveness + also compute on decimated/full via rfft)
    freqs_full, mag_full = fft_spectrum(y[::1], fs)  # full
    dom = dominant_frequencies(freqs_full, mag_full, top_n=5, fmin=1.0)
    for i,d in enumerate(dom,1):
        st[f'dom_freq_{i}_hz'] = d['frequency_hz']
        st[f'dom_mag_{i}'] = d['magnitude']
    records.append(st)
    # store short segment for plots (middle) and PSD
    nseg = min(len(y), int(MAX_PLOT_SEC*fs))
    mid = len(y)//2
    seg = y[mid:mid+nseg].copy()
    f_psd, pxx = signal.welch(y, fs=fs, nperseg=PSD_NPERSEG)
    f_stft, t_stft, Z = signal.stft(y[:min(len(y), int(2*fs))], fs=fs, nperseg=1024, noverlap=768)  # 2s window for spectrogram
    segments[p.name] = {'seg': seg, 'fs': fs, 'unit': unit, 'f_psd': f_psd, 'pxx': pxx, 'f_stft': f_stft, 't_stft': t_stft, 'Zxx': Z}

stats_df = pd.DataFrame(records)
stats_df


## 3. Raw waveform plots (short time segments)

Short middle segments shown for readability; full durations/units are verified per inventory.


In [ ]:
n = len(files)
cols = 2
rows = int(np.ceil(n/cols))
fig, axes = plt.subplots(rows, cols, figsize=(12, 3*rows), squeeze=False)
for i, p in enumerate(files):
    ax = axes[i//cols, i%cols]
    s = segments[p.name]
    t = np.arange(len(s['seg']))/s['fs']
    ax.plot(t, s['seg'], lw=0.6)
    ax.set_title(p.name)
    ax.set_xlabel('Time (s) [verified x-axis unit]' if s['unit'] else 'Time (s)')
    ax.set_ylabel(f"Amplitude ({s['unit']})" if s['unit'] else 'Amplitude (unit unverified)')
for j in range(i+1, rows*cols):
    axes[j//cols, j%cols].axis('off')
plt.tight_layout(); plt.savefig(out_fig/'waveforms.png', dpi=150); plt.show()


## 4. Signal amplitude distributions


In [ ]:
fig, axes = plt.subplots(rows, cols, figsize=(12, 3*rows), squeeze=False)
for i, p in enumerate(files):
    ax = axes[i//cols, i%cols]
    y,_ = load_acoustic_values(p)  # sequential load
    s = segments[p.name]
    ax.hist(y, bins=120, color='#4c78a8', alpha=0.85)
    ax.set_title(p.name)
    ax.set_xlabel(f"Amplitude ({s['unit']})" if s['unit'] else 'Amplitude')
    ax.set_ylabel('Count')
for j in range(i+1, rows*cols):
    axes[j//cols, j%cols].axis('off')
plt.tight_layout(); plt.savefig(out_fig/'amplitude_distributions.png', dpi=150); plt.show()


## 5. Time-domain statistics

RMS, standard deviation, peak-to-peak, kurtosis (Fisher), skewness, crest factor. Computed per full verified recording.


In [ ]:
cols_show = ['file','y_unit','rms','std','peak_to_peak','kurtosis_fisher','skewness','crest_factor','mean','peak_abs']
stats_df[cols_show]


In [ ]:
metrics = ['rms','std','peak_to_peak','kurtosis_fisher','skewness','crest_factor']
fig, axes = plt.subplots(2, 3, figsize=(14,7))
for ax, m in zip(axes.ravel(), metrics):
    ax.bar(stats_df['file'], stats_df[m], color='#72b7b2')
    ax.set_title(m)
    ax.tick_params(axis='x', rotation=45)
    if m in ['rms','std','peak_to_peak','peak_abs']:
        ax.set_ylabel(f"Value ({stats_df['y_unit'].iloc[0]})" if stats_df['y_unit'].nunique()==1 else 'Value (verified unit)')
    else:
        ax.set_ylabel('Dimensionless')
plt.tight_layout(); plt.savefig(out_fig/'time_domain_stats.png', dpi=150); plt.show()


## 6. FFT analysis and dominant frequencies

Single-sided FFT magnitude (Hann window, amplitude-scaled) on verified sampling frequency; frequency axis in Hz (verified).


In [ ]:
fig, axes = plt.subplots(rows, cols, figsize=(12, 3*rows), squeeze=False)
for i, p in enumerate(files):
    ax = axes[i//cols, i%cols]
    y, meta = load_acoustic_values(p)
    freqs, mag = fft_spectrum(y, meta['fs_hz'])
    # plot up to 10 kHz for visibility (fs/2=25.6kHz)
    msk = freqs <= 10000
    ax.semilogy(freqs[msk], mag[msk]+1e-12, lw=0.6)
    ax.set_title(p.name)
    ax.set_xlabel('Frequency (Hz) [verified]')
    ax.set_ylabel(f"Magnitude ({meta['y_unit']})" if meta['y_unit'] else 'Magnitude')
for j in range(i+1, rows*cols):
    axes[j//cols, j%cols].axis('off')
plt.tight_layout(); plt.savefig(out_fig/'fft.png', dpi=150); plt.show()

dom_cols = ['file'] + [c for c in stats_df.columns if c.startswith('dom_freq_') or c.startswith('dom_mag_')]
stats_df[dom_cols]


## 7. Power spectral density (PSD)

Welch PSD with verified sampling frequency (Hz) and amplitude unit (`Pa`) where verified.


In [ ]:
fig, axes = plt.subplots(rows, cols, figsize=(12, 3*rows), squeeze=False)
for i, p in enumerate(files):
    ax = axes[i//cols, i%cols]
    s = segments[p.name]
    msk = s['f_psd'] <= 10000
    ax.semilogy(s['f_psd'][msk], s['pxx'][msk]+1e-20, lw=0.7)
    ax.set_title(p.name)
    ax.set_xlabel('Frequency (Hz) [verified]')
    ax.set_ylabel(f"PSD ({s['unit']}^2/Hz)" if s['unit'] else 'PSD (unit^2/Hz)')
for j in range(i+1, rows*cols):
    axes[j//cols, j%cols].axis('off')
plt.tight_layout(); plt.savefig(out_fig/'psd.png', dpi=150); plt.show()


## 8. STFT spectrograms

Computed on a 2-second window per file for visibility (sequential processing). Axes in seconds/Hz (verified).


In [ ]:
fig, axes = plt.subplots(rows, cols, figsize=(12, 3*rows), squeeze=False)
for i, p in enumerate(files):
    ax = axes[i//cols, i%cols]
    s = segments[p.name]
    Z = np.abs(s['Zxx'])
    Zdb = 20*np.log10(Z + 1e-12)
    im = ax.pcolormesh(s['t_stft'], s['f_stft'], Zdb, shading='gouraud', cmap='magma')
    ax.set_ylim(0, 10000)
    ax.set_title(p.name)
    ax.set_xlabel('Time (s) [verified]')
    ax.set_ylabel('Frequency (Hz) [verified]')
    fig.colorbar(im, ax=ax, label=f"dB re 1 {s['unit']} (relative magnitude)" if s['unit'] else 'dB (relative)')
for j in range(i+1, rows*cols):
    axes[j//cols, j%cols].axis('off')
plt.tight_layout(); plt.savefig(out_fig/'spectrograms.png', dpi=150); plt.show()


## 9. Normal versus faulty recording comparisons

Comparisons across verified metrics and spectra (Normal vs BPFI/BPFO at 0Nm, severities 03/10 where present).


In [ ]:
# Merge labels
meta_cols = inv[['filename','fault_category','operating_condition_load','severity_or_variant']].rename(columns={'filename':'file'})
cmp_df = stats_df.merge(meta_cols, on='file', how='left')
order = ['Normal','BPFO','BPFI']
fig, axes = plt.subplots(2, 3, figsize=(14,7))
for ax, m in zip(axes.ravel(), ['rms','kurtosis_fisher','crest_factor','peak_to_peak','std','skewness']):
    for fc in order:
        sub = cmp_df[cmp_df['fault_category']==fc]
        ax.scatter([fc]*len(sub), sub[m], label=fc, s=60)
        for _,r in sub.iterrows():
            ax.annotate(r.get('severity_or_variant',''), (fc, r[m]), textcoords='offset points', xytext=(4,4), fontsize=8)
    ax.set_title(f'{m}: Normal vs Faulty')
    ax.set_ylabel(f"{m} ({cmp_df['y_unit'].iloc[0]})" if m in ['rms','std','peak_to_peak'] and cmp_df['y_unit'].nunique()==1 else m)
plt.tight_layout(); plt.savefig(out_fig/'normal_vs_faulty_stats.png', dpi=150); plt.show()

# Spectral comparison overlay (PSD)
plt.figure(figsize=(10,5))
for p in files:
    s = segments[p.name]
    msk = s['f_psd'] <= 5000
    plt.semilogy(s['f_psd'][msk], s['pxx'][msk]+1e-20, label=p.name, lw=1)
plt.xlabel('Frequency (Hz) [verified]'); plt.ylabel('PSD (Pa^2/Hz) [verified]' if True else 'PSD')
plt.title('PSD comparison: Normal vs Faulty recordings')
plt.legend(fontsize=8); plt.tight_layout(); plt.savefig(out_fig/'normal_vs_faulty_psd.png', dpi=150); plt.show()
cmp_df[['file','fault_category','severity_or_variant','rms','kurtosis_fisher','crest_factor','dom_freq_1_hz','dom_mag_1']].sort_values(['fault_category','severity_or_variant'])


## 10. Feature distribution plots


In [ ]:
feat = cmp_df.copy()
fig, axes = plt.subplots(2, 3, figsize=(14,7))
for ax, m in zip(axes.ravel(), ['rms','kurtosis_fisher','crest_factor','skewness','peak_to_peak','std']):
    for name, g in feat.groupby('fault_category'):
        ax.hist(g[m].values, alpha=0.6, label=name, bins=5)
    ax.set_title(f'Distribution: {m}')
    ax.legend()
plt.tight_layout(); plt.savefig(out_fig/'feature_distributions.png', dpi=150); plt.show()

# pairwise scatter
plt.figure(figsize=(6,5))
for name, g in feat.groupby('fault_category'):
    plt.scatter(g['rms'], g['kurtosis_fisher'], label=name, s=70)
    for _,r in g.iterrows():
        plt.annotate(r['severity_or_variant'] if pd.notna(r['severity_or_variant']) else r['file'], (r['rms'], r['kurtosis_fisher']), fontsize=8)
plt.xlabel(f"RMS ({feat['y_unit'].iloc[0]})" if feat['y_unit'].nunique()==1 else 'RMS')
plt.ylabel('Kurtosis (Fisher, dimensionless)')
plt.title('RMS vs Kurtosis by fault category')
plt.legend(); plt.tight_layout(); plt.savefig(out_fig/'rms_vs_kurtosis.png', dpi=150); plt.show()


## 11. Data-quality observations

- File/structure consistency, verified metadata coverage, and any warnings (including duplicate-metadata heuristic flags) are summarized below. Invalid/corrupted recordings must not be silently discarded; all inspected files are represented in inventory.


In [ ]:
quality = inv[['filename','inspection_status','format','matlab_structure','signal_field','signal_shape','data_type','sampling_frequency_hz_verified','duration_s_verified','x_unit_verified','units_verified','warnings']].copy()
print('Status counts:', inv['inspection_status'].value_counts(dropna=False).to_dict())
print('Formats:', inv['format'].value_counts(dropna=False).to_dict())
# flatten warnings
for _,r in quality.iterrows():
    print(r['filename'], '->', r['warnings'])
quality


## 12. Summary of EDA outputs

- Inventory: `outputs/acoustic_inventory.csv`, `outputs/acoustic_inventory.json`, `outputs/acoustic_inventory_summary.json`
- Figures: `outputs/figures/*.png`
- This notebook is EDA-only (no train/test split, no model training). Any unverified metadata remains explicitly marked in inventory/report.


In [ ]:
# Save computed stats for report
stats_out = EDA_DIR / 'outputs' / 'acoustic_time_domain_stats.csv'
stats_df.merge(inv[['filename','fault_category','operating_condition_load','severity_or_variant']].rename(columns={'filename':'file'}), on='file', how='left').to_csv(stats_out, index=False)
print('Saved:', stats_out)
